<a href="https://colab.research.google.com/github/Vaishnavi-tomar/AI-INVENTORY_ENGINE/blob/main/Inventory_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Run this first. It takes about 1 minute.
!pip install git+https://github.com/amazon-science/chronos-forecasting.git
!pip install gradio fastapi uvicorn nest-asyncio

  Cloning https://github.com/amazon-science/chronos-forecasting.git to /tmp/pip-req-build-ukxw4pzb
  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git /tmp/pip-req-build-ukxw4pzb
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit 6d68ed7c4ed2805d122d77b4660765b4089de5ca
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
import pandas as pd
import zipfile
from google.colab import drive

# 1. Connect to Drive (Click 'Allow' in the popup)
drive.mount('/content/drive')

# 2. Path to your folder
drive_path = '/content/drive/MyDrive/M5_Project/'

# 3. Unzip directly to the new local runtime
with zipfile.ZipFile('/content/sales_train_evaluation.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

with zipfile.ZipFile('/content/sell_prices.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

# 4. Load the data
sales = pd.read_csv('/content/sales_train_evaluation.csv', nrows=2000)
calendar = pd.read_csv('/content/calendar.csv')
prices = pd.read_csv('/content/sell_prices.csv', nrows=5000)

print("✅ Data is back and unzipped!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Data is back and unzipped!


In [12]:
import numpy as np

# 1. Melt Sales
df = pd.melt(sales, id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
             var_name='d', value_name='sales')

# 2. Merge Calendar & Prices
df = df.merge(calendar[['d', 'event_name_1', 'wm_yr_wk']], on='d', how='left')
df = df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

# 3. Clean up
df['sell_price'] = df['sell_price'].fillna(df.groupby('id')['sell_price'].transform('mean'))
df['is_event'] = df['event_name_1'].notnull().astype(int)

print("✅ Master Table is Re-created and ready for the AI!")

✅ Master Table is Re-created and ready for the AI!


In [13]:
import torch
from chronos import ChronosPipeline

# 1. Load the "Base" version of the model (Free and fast for Colab)
# We use 'cuda' because we turned on the T4 GPU earlier!
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)

print("✅ AI Brain Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/806M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

✅ AI Brain Loaded Successfully!


In [15]:
import torch
from chronos import ChronosPipeline

# 1. Load the "Base" version of the model (Free and fast for Colab)
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",
    device_map="cuda",  # This uses the Colab GPU for speed!
    torch_dtype=torch.bfloat16,
)

# 2. Pick one product from your Master Table (e.g., the first item)
# We take the last 100 days of sales to "show" the model the trend
context = torch.tensor(df[df['id'] == df['id'].unique()[0]]['sales'].values[-100:])

# 3. Predict the next 28 days (The standard Amazon planning window)
forecast = pipeline.predict(context, prediction_length=28)

print("Forecast Complete! The 'Brain' has spoken.")

Forecast Complete! The 'Brain' has spoken.


In [14]:
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np

def inventory_engine(product_id, current_stock):
    # 1. Get the history for the selected product (last 100 days)
    context_data = df[df['id'] == product_id]['sales'].values[-100:]
    context_tensor = torch.tensor(context_data)

    # 2. Ask the AI to predict the next 21 days
    forecast = pipeline.predict(context_tensor, prediction_length=21)

    # 3. Calculate the "Risk Zone" (P10, P50, P90)
    low, median, high = np.quantile(forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0)

    # 4. Supply Chain Logic: The Reorder Point (ROP)
    # If high-end demand (P90) over next 7 days > current stock, we need more!
    rop = np.sum(high[:7])

    if current_stock <= 0:
        status = "🔴 OUT OF STOCK"
    elif current_stock <= rop:
        status = f"⚠️ REORDER IMMEDIATELY (Threshold: {int(rop)})"
    else:
        status = "🟢 STOCK HEALTHY"

    # 5. Create the Visualization
    plt.figure(figsize=(10, 4))
    plt.plot(context_data[-30:], label="Past 30 Days Sales", color="black")
    plt.plot(np.arange(30, 51), median, label="Predicted Sales", color="blue")
    plt.fill_between(np.arange(30, 51), low, high, color="blue", alpha=0.2, label="Risk Zone")
    plt.axhline(y=current_stock, color='red', linestyle='--', label="Current Stock Level")
    plt.legend()
    plt.title(f"Inventory Forecast: {product_id}")

    return status, plt.gcf()

# 6. Launch the App with a Public Link
demo = gr.Interface(
    fn=inventory_engine,
    # We show the first 50 unique products in the dropdown
    inputs=[
        gr.Dropdown(choices=list(df['id'].unique()[:50]), label="Select Product"),
        gr.Number(label="Enter Current Warehouse Stock", value=50)
    ],
    outputs=[
        gr.Textbox(label="Action Plan"),
        gr.Plot(label="Demand Visualizer")
    ],
    title="📦 Amazon-Style Stock-Run Engine",
    description="Using Transformer-based AI to prevent out-of-stock events."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b25fa0118dcd8ef56c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
# 1. Define Lead Time (7 days for a supplier to ship to Walmart)
lead_time = 7

def calculate_rop(median_forecast, high_forecast):
    # Expected demand during the time we wait for the truck
    demand_during_lead_time = np.sum(median_forecast[:lead_time])

    # Safety Buffer = (High-end Risk - Average Demand)
    # This protects us if a sudden "Stock Run" happens
    safety_stock = np.sum(high_forecast[:lead_time]) - demand_during_lead_time

    return demand_during_lead_time + safety_stock

print("✅ Day 12 Logic Updated: Lead-Time Demand + Safety Stock.")

✅ Day 12 Logic Updated: Lead-Time Demand + Safety Stock.


In [18]:
import pandas as pd
import numpy as np
import torch

# 1. Re-define the Prediction Logic for the Batch
def get_prediction(product_id):
    # Get last 100 days of sales for context
    data = df[df['id'] == product_id]['sales'].values[-100:]
    context = torch.tensor(data)

    # Run the Chronos Brain (make sure 'pipeline' was loaded in Day 7)
    forecast = pipeline.predict(context, prediction_length=21)

    # Calculate Probabilities
    low, median, high = np.quantile(forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0)
    return data, low, median, high

# 2. Re-define the Reorder Logic (Day 12)
def calculate_rop(median_f, high_f):
    lead_time = 7
    demand_during_lt = np.sum(median_f[:lead_time])
    safety_stock = np.sum(high_f[:lead_time]) - demand_during_lt
    return demand_during_lt + safety_stock

# 3. Run the Batch for 50 Items
print("🚀 Starting Batch Process for 50 items... please wait.")
results = []
top_items = df['id'].unique()[:50]

for item in top_items:
    try:
        history, low, median, high = get_prediction(item)
        rop = calculate_rop(median, high)

        # Simulate a current stock level to test the logic
        current_inv = np.random.randint(10, 100)

        results.append({
            "Product_ID": item,
            "Current_Stock": current_inv,
            "Reorder_Point": int(rop),
            "Status": "🚨 REORDER" if current_inv <= rop else "✅ HEALTHY"
        })
    except Exception as e:
        print(f"Error processing {item}: {e}")

# 4. Save the Final Report
report_df = pd.DataFrame(results)
report_df.to_csv("Amazon_Inventory_Report_2026.csv", index=False)

print("✅ Day 13 Success! Download 'Amazon_Inventory_Report_2026.csv' from your sidebar.")
report_df.head()

🚀 Starting Batch Process for 50 items... please wait.
✅ Day 13 Success! Download 'Amazon_Inventory_Report_2026.csv' from your sidebar.


,Product_ID,Current_Stock,Reorder_Point,Status
0,HOBBIES_1_001_CA_1_evaluation,40,21,✅ HEALTHY
1,HOBBIES_1_002_CA_1_evaluation,92,4,✅ HEALTHY
2,HOBBIES_1_003_CA_1_evaluation,25,14,✅ HEALTHY
3,HOBBIES_1_004_CA_1_evaluation,74,27,✅ HEALTHY
4,HOBBIES_1_005_CA_1_evaluation,25,32,🚨 REORDER


In [20]:
with open('requirements.txt', 'w') as f:
    f.write('pandas\nnumpy\ntorch\nchronos-forecasting\ngradio\nfastapi\nmatplotlib')
print("✅ requirements.txt generated!")

✅ requirements.txt generated!
